# 02 — Auditoria e qualidade dos dados

Avalia tipos, nulos, duplicidades, consistência temporal, coordenadas e campos de vítimas.

In [ ]:
import os
import pandas as pd
import numpy as np

PASTA_PROJETO = "TCC_PRF"
PASTA_PROCESSED = os.path.join(PASTA_PROJETO, "data", "processed")
arquivo_base = os.path.join(PASTA_PROCESSED, "prf_acidentes_2020_2025.parquet")
df = pd.read_parquet(arquivo_base)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

In [ ]:
def exemplo_valor(serie):
    valores = serie.dropna()
    return None if len(valores) == 0 else valores.iloc[0]

auditoria = pd.DataFrame({
    "coluna": df.columns,
    "tipo": [str(df[c].dtype) for c in df.columns],
    "registros": [len(df)] * len(df.columns),
    "nulos": [df[c].isna().sum() for c in df.columns],
    "percentual_nulos": [round(df[c].isna().mean() * 100, 4) for c in df.columns],
    "valores_unicos": [df[c].nunique(dropna=True) for c in df.columns],
    "exemplo": [exemplo_valor(df[c]) for c in df.columns]
}).sort_values("percentual_nulos", ascending=False)

display(auditoria)

In [ ]:
print("Duplicados completos:", df.duplicated().sum())
print("IDs duplicados:", df["id"].duplicated().sum())

data_convertida = pd.to_datetime(df["data_inversa"], errors="coerce")
print("Datas inválidas:", data_convertida.isna().sum())
print("Período:", data_convertida.min(), "a", data_convertida.max())
print("Ano da data diferente de ano_fonte:",
      ((data_convertida.dt.year != df["ano_fonte"]) & data_convertida.notna()).sum())

horario_convertido = pd.to_datetime(df["horario"], format="%H:%M:%S", errors="coerce")
print("Horários inválidos:", horario_convertido.isna().sum())

In [ ]:
lat = pd.to_numeric(df["latitude"], errors="coerce")
lon = pd.to_numeric(df["longitude"], errors="coerce")
coords_invalidas = (~(lat.between(-34, 6) & lon.between(-74, -34))) & lat.notna() & lon.notna()
print("Coordenadas fora dos limites aproximados do Brasil:", coords_invalidas.sum())

for c in ["pessoas","mortos","feridos_leves","feridos_graves","ilesos","ignorados","feridos","veiculos"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("feridos != leves + graves:",
      (df["feridos"] != (df["feridos_leves"] + df["feridos_graves"])).sum())

soma_pessoas = (
    df["mortos"].fillna(0) + df["feridos_leves"].fillna(0) +
    df["feridos_graves"].fillna(0) + df["ilesos"].fillna(0) +
    df["ignorados"].fillna(0)
)
print("pessoas != soma das classificações:",
      (df["pessoas"].notna() & (df["pessoas"] != soma_pessoas)).sum())

In [ ]:
auditoria.to_csv(
    os.path.join(PASTA_PROCESSED, "auditoria_qualidade_prf.csv"),
    index=False, encoding="utf-8-sig"
)
print("Auditoria salva.")